# Fine-tuning DistilBERT for Text Classification

This notebook outlines the steps to fine-tune a DistilBERT model for a text classification task, using data prepared by your teammate, and then uploading the fine-tuned adapters to Hugging Face.

## 1. Setup: Install and Import Libraries

First, we need to install the `transformers` library for working with pre-trained models and `datasets` for efficient data handling. We'll also include `accelerate` for optimized training and `evaluate` for metric computation.

In [1]:
!pip install -qqq transformers datasets accelerate evaluate peft "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.2 MB/s eta 0:00:00


In [2]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
from peft import get_peft_model, LoraConfig, TaskType

## 2. Data Loading and Preprocessing

This section is crucial. You'll need to load your data into a format that can be processed by the `datasets` library. Typically, this involves loading a CSV, JSON, or similar file that contains your text and corresponding labels.

**Action Required:** Please adapt the `load_your_data` function below to correctly load the data from your teammate's notebook. Look at `https://github.com/NateyLB/CMPE255Project/blob/main/notebooks/02_Data_Preprocessing.ipynb` to understand their data structure and output format. Ensure your data has 'text' and 'label' columns, or adjust the code accordingly.

For demonstration, I've included a placeholder for dummy data.

In [3]:
import pandas as pd
from pathlib import Path
from google.colab import drive
from datasets import Dataset, DatasetDict

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# --- EDIT THIS LINE if your folder is named differently ---
DRIVE_ROOT = Path('/content/drive/MyDrive/Guna_CMPE255_Project_Final')
# ----------------------------------------------------------

DATA_DIR = DRIVE_ROOT / 'data'

def load_your_data():
    """Load train/test parquet files from Google Drive."""
    train_path = DATA_DIR / 'train.parquet'
    test_path  = DATA_DIR / 'test.parquet'

    print(f'Loading training data from: {train_path}')
    print(f'Loading test data from: {test_path}')

    df_train = pd.read_parquet(train_path)
    df_test  = pd.read_parquet(test_path)

    print(f'Train samples: {len(df_train)}')
    print(f'Test samples:  {len(df_test)}')

    # Convert pandas DataFrames to Hugging Face Datasets
    train_dataset = Dataset.from_pandas(df_train)
    test_dataset  = Dataset.from_pandas(df_test)

    # Create a DatasetDict
    # Use 10% of training data for validation during fine-tuning
    train_val_split = train_dataset.train_test_split(test_size=0.1, seed=42)

    dataset_dict = DatasetDict({
        'train':      train_val_split['train'],
        'validation': train_val_split['test'],
        'test':       test_dataset,
    })

    return dataset_dict

# Load the dataset
dataset_dict = load_your_data()

print("\nDataset loaded and split:")
print(dataset_dict)
print("\nExample from training set:")
print(dataset_dict['train'][0])

# Automatically map string labels to integer IDs (alphabetically sorted)
unique_labels = sorted(list(set(dataset_dict['train']['label'])))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"\nLabel mapping created: {label2id}")

# Apply the mapping to ensure labels are integers
dataset_dict = dataset_dict.map(lambda examples: {'label': [label2id[l] for l in examples['label']]}, batched=True)
print("Labels successfully converted to integers.")

Mounted at /content/drive
Loading training data from: /content/drive/MyDrive/Guna_CMPE255_Project_Final/data/train.parquet
Loading test data from: /content/drive/MyDrive/Guna_CMPE255_Project_Final/data/test.parquet
Train samples: 6034
Test samples:  10537

Dataset loaded and split:
DatasetDict({
    train: Dataset({
        features: ['statement', 'label'],
        num_rows: 5430
    })
    validation: Dataset({
        features: ['statement', 'label'],
        num_rows: 604
    })
    test: Dataset({
        features: ['statement', 'label'],
        num_rows: 10537
    })
})

Example from training set:
{'statement': 'Sleeping 6 hours, heart Racing and chest pain Hello all.\n\nI have been on sick leave from work for 3 months now. \nTomorrow I’m starting slowly again. 3 hours. \n\nThroughout all January I have had this persistent restlessness. I have been sleeping 4-6 hours every night, woken up, and slept again for 2-4 hours. Getting up at noon. The past two weeks I have tried to get u

Map:   0%|          | 0/5430 [00:00<?, ? examples/s]

Map:   0%|          | 0/604 [00:00<?, ? examples/s]

Map:   0%|          | 0/10537 [00:00<?, ? examples/s]

Labels successfully converted to integers.


## 3. Tokenization

To prepare the text data for DistilBERT, we need to convert it into a format that the model can understand. This involves tokenizing the text and mapping tokens to their corresponding IDs. We'll use the `DistilBertTokenizerFast` for this purpose.

**Important:** Ensure your text column is named 'text' and label column is named 'label' in your `dataset_dict`. If not, you might need to adjust the `tokenize_function` or rename your columns beforehand.

In [4]:
print("\nFeatures in the training dataset:")
print(dataset_dict['train'].features)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    # Use 'statement' column which contains the raw text
    return tokenizer(examples['statement'], truncation=True, max_length=512)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

print("Tokenization complete!")
print("Example tokenized entry from training set:")
print(tokenized_datasets['train'][0])


Features in the training dataset:
{'statement': Value('string'), 'label': Value('int64')}


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5430 [00:00<?, ? examples/s]

Map:   0%|          | 0/604 [00:00<?, ? examples/s]

Map:   0%|          | 0/10537 [00:00<?, ? examples/s]

Tokenization complete!
Example tokenized entry from training set:
{'statement': 'Sleeping 6 hours, heart Racing and chest pain Hello all.\n\nI have been on sick leave from work for 3 months now. \nTomorrow I’m starting slowly again. 3 hours. \n\nThroughout all January I have had this persistent restlessness. I have been sleeping 4-6 hours every night, woken up, and slept again for 2-4 hours. Getting up at noon. The past two weeks I have tried to get up at 8, to match my work schedule. Now I only sleep 6 hours and stay in bed trying to get in the last two hours. \nIt worries me that I’m still not sleeping through a full night.\nI’m not sure if it’s stress or my hiatal hernia or my chronic neck pain that wakes me up. But I wake up rather abruptly. So I’m guessing stress. Sometimes I wake up with racing heart.\n\nSpeaking of racing heart - I started to get that randomly racing heart throughout the day. I wonder if it’s because I’m starting work and I’m nervous about it. I have social anxi

## 4. Model Loading and Metrics Definition

We'll load the pre-trained `DistilBertForSequenceClassification` model and define a function to compute evaluation metrics during training, specifically accuracy.

In [5]:
num_labels = len(unique_labels)  # 7 classes
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Configure LoRA
# CRITICAL: modules_to_save ensures the classification head (pre_classifier + classifier)
# is saved alongside the LoRA adapters. Without this, loading the model later
# results in a randomly-initialized classifier head and terrible performance.
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,                # increased rank for more capacity (was 8)
    lora_alpha=32,       # scale factor (typically 2x rank)
    lora_dropout=0.1,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],  # all attention projections
    modules_to_save=["pre_classifier", "classifier"],  # ← SAVES THE CLASSIFICATION HEAD
)

# Wrap the base model with the LoRA configuration
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = evaluate.load('accuracy').compute(predictions=predictions, references=labels)
    f1 = evaluate.load('f1').compute(predictions=predictions, references=labels, average='macro')
    return {'accuracy': acc['accuracy'], 'macro_f1': f1['f1']}

print(f"\nModel loaded with {num_labels} labels and wrapped with LoRA adapters.")
print(f"Classification head (pre_classifier + classifier) will be saved with adapters.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,185,799 || all params: 68,144,654 || trainable%: 1.7401

Model loaded with 7 labels and wrapped with LoRA adapters.
Classification head (pre_classifier + classifier) will be saved with adapters.


## 5. Configure Training Arguments and Trainer

Here we define the `TrainingArguments`, which specify hyperparameters and settings for the training process. Then, we initialize the `Trainer` with our model, training arguments, datasets, tokenizer, and metrics function.

In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
)

print("Trainer initialized.")
print("Training for 10 epochs with lr=5e-5 and warmup_ratio=0.1")
print("Best model will be selected by validation macro_f1.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer initialized.
Training for 10 epochs with lr=5e-5 and warmup_ratio=0.1
Best model will be selected by validation macro_f1.


## 6. Train the Model

Now, let's start the fine-tuning process. This might take some time depending on your dataset size and hardware.

In [7]:
from transformers import DataCollatorWithPadding

print("Starting model training...")

# Add a data collator to pad sequences dynamically to the batch's max length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer.data_collator = data_collator

trainer.train()
print("Training complete!")

# Evaluate the model on the test set
results = trainer.evaluate(tokenized_datasets['test'])
print("\nEvaluation results on test set:")
print(results)

Starting model training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.295969,0.569536,0.564879
2,1.549827,0.811376,0.693709,0.690089
3,0.783525,0.739335,0.738411,0.733864
4,0.783525,0.675695,0.758278,0.755554
5,0.651353,0.654577,0.758278,0.760371
6,0.574075,0.625800,0.771523,0.769859
7,0.574075,0.627089,0.773179,0.771190
8,0.539662,0.621892,0.774834,0.773193
9,0.520267,0.612057,0.774834,0.773849
10,0.520267,0.606638,0.783113,0.783079


Training complete!



Evaluation results on test set:
{'eval_loss': 0.6211133003234863, 'eval_accuracy': 0.7568567903577869, 'eval_macro_f1': 0.7142296893947986, 'eval_runtime': 14.7068, 'eval_samples_per_second': 716.47, 'eval_steps_per_second': 44.809, 'epoch': 10.0}


## 7. Upload Fine-tuned Adapters to Hugging Face

In [8]:
# ============================================================
# Save Fine-tuned Model (LoRA adapters + classification head)
# ------------------------------------------------------------
# Because modules_to_save=["pre_classifier", "classifier"] was set,
# save_pretrained() will now include the trained classification head
# weights alongside the LoRA adapter matrices.
# ============================================================

# Save locally
LOCAL_SAVE_DIR = "distilBERT_mentalhealth_detection"
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)
print(f"Model saved locally to: {LOCAL_SAVE_DIR}")

# Also save to Google Drive for persistence
DRIVE_MODEL_DIR = DRIVE_ROOT / 'models' / 'distilbert_lora'
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(DRIVE_MODEL_DIR))
tokenizer.save_pretrained(str(DRIVE_MODEL_DIR))
print(f"Model saved to Drive: {DRIVE_MODEL_DIR}")

# Push to HuggingFace Hub (optional — uncomment if you have HF_TOKEN set)
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')
# model.push_to_hub("NathanSJSU01/distilBERT_mentalhealth_detection", token=HF_TOKEN)
# tokenizer.push_to_hub("NathanSJSU01/distilBERT_mentalhealth_detection", token=HF_TOKEN)
# print("Model pushed to HuggingFace Hub.")

print("\nDone! The model now includes the classification head weights.")
print("NB-08 can load this model and get proper predictions.")

Model saved locally to: distilBERT_mentalhealth_detection
Model saved to Drive: /content/drive/MyDrive/Guna_CMPE255_Project_Final/models/distilbert_lora

Done! The model now includes the classification head weights.
NB-08 can load this model and get proper predictions.
